# NBA Position Evolution Analysis (2015-16 -- 2025-26)
### Are traditional positions still predictive of how players actually play?

**Research questions**

1. Have traditional positions (PG/SG/SF/PF/C) become less predictive of a player's statistical profile over time?
2. How have data-driven player archetypes evolved from 2015-16 through 2025-26?

**Data source:** `nba_api` (`leaguedashplayerstats`, `playerindex`) -- Regular Season only.

**Sample restriction:** `GP >= 20` and `MIN >= 15` (At least 20 games played, with at least 15 minutes played per game), applied after retrieval.

We first start by doing our initial setup for data collection, mainly establishing our years (2015-2026) and creating our folder directories.

In [ ]:
import pandas as pd
from pathlib import Path



# Creating directories for raw and processed data 
current_folder = Path("/home/jaime/Documents/coding-projects/school/NBA-Roles-Research") # change this according to stored root directory
DATA_DIR = current_folder / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
for directory in [DATA_DIR, RAW_DATA_DIR, PROCESSED_DATA_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


# Establishing our years (2015-2026)
START_YEAR = 2015
END_YEAR = 2025
SEASONS = [f"{year}-{str(year + 1)[-2:]}" for year in range(START_YEAR, END_YEAR)]
print(f"{len(SEASONS)} seasons: {SEASONS}")

10 seasons: ['2015-16', '2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25']


## 1. Data Collection

### Endpoints used (and why)

| Need | Endpoint | Notes |
|---|---|---|
| Per-game counting stats (PTS, REB, AST, STL, BLK, FG3M/A, TOV, MIN, ...) | `leaguedashplayerstats(measure_type_detailed_defense="Base", per_mode_detailed="PerGame")` | Raw per-game volume is needed for several EDA questions. |
| Rate/percentage stats (USG%, AST%, REB%, TS%, ...) | `leaguedashplayerstats(measure_type_detailed_defense="Advanced", per_mode_detailed="PerGame")` | Percentage-based features help to eliminate potential bias from time spent in the game. |
| Traditional position label | `playerindex(season=..., historical_nullable="1")` | Returns a `POSITION` field **per season**, which is what we want. |


In [21]:
from nba_api.stats.endpoints import leaguedashplayerstats, playerindex
import time

REQUEST_SLEEP_SEC = 0.7

def _cached_csv(path: Path, fetch_fn):
    """Try to read a CSV from disk, otherwise fetch it and save it."""
    if path.exists():
        return pd.read_csv(path)
    df = fetch_fn()
    df.to_csv(path, index=False)
    time.sleep(REQUEST_SLEEP_SEC)
    return df


######################
# FETCHING FUNCTIONS #
######################


def fetch_base_stats(season: str) -> pd.DataFrame:
    resp = leaguedashplayerstats.LeagueDashPlayerStats(
        season=season,
        season_type_all_star="Regular Season",
        measure_type_detailed_defense="Base",
        per_mode_detailed="PerGame",
    )
    df = resp.get_data_frames()[0]
    df["SEASON"] = season
    return df

def fetch_advanced_stats(season: str) -> pd.DataFrame:
    resp = leaguedashplayerstats.LeagueDashPlayerStats(
        season=season,
        season_type_all_star="Regular Season",
        measure_type_detailed_defense="Advanced",
        per_mode_detailed="PerGame",
    )
    df = resp.get_data_frames()[0]
    df["SEASON"] = season
    return df

def fetch_positions(season: str) -> pd.DataFrame:
    resp = playerindex.PlayerIndex(season=season, historical_nullable="1")
    df = resp.get_data_frames()[0]
    df = df[["PERSON_ID", "POSITION"]].rename(columns={"PERSON_ID": "PLAYER_ID"})
    df["SEASON"] = season
    return df


####################
# GETTER FUNCTIONS #
####################
# These functions will check if data already exists (by calling _cached_csv); 
# if not, they will fetch it (from the fetch functions) and save it to disk.

def get_season_base(season):
    return _cached_csv(RAW_DATA_DIR / f"base_{season}.csv", lambda: fetch_base_stats(season))

def get_season_advanced(season):
    return _cached_csv(RAW_DATA_DIR / f"advanced_{season}.csv", lambda: fetch_advanced_stats(season))

def get_season_positions(season):
    return _cached_csv(RAW_DATA_DIR / f"positions_{season}.csv", lambda: fetch_positions(season))

With our functions defined, we can run the following functions in order to build our data with our variables previously defined

In [22]:
base_frames, advanced_frames, position_frames = [], [], []
for season in SEASONS:
    base_frames.append(get_season_base(season))
    advanced_frames.append(get_season_advanced(season))
    position_frames.append(get_season_positions(season))

base_raw = pd.concat(base_frames, ignore_index=True) if base_frames else pd.DataFrame()
advanced_raw = pd.concat(advanced_frames, ignore_index=True) if advanced_frames else pd.DataFrame()
position_raw = pd.concat(position_frames, ignore_index=True) if position_frames else pd.DataFrame()

print(base_raw.shape, advanced_raw.shape, position_raw.shape)

(5386, 68) (5386, 80) (51861, 3)


With our data now established, we can now begin building a "master" dataframe out of the 3 dataframes we have already collected.

We will do this by focusing on the columns we want to keep. Specifically those in the "ADV_KEEP" list.

In [23]:
# Building a smaller dataframe of advanced stats to merge with base stats
ADV_KEEP = [
    "PLAYER_ID", "SEASON", "USG_PCT", "AST_PCT", "AST_TO", "AST_RATIO",
    "OREB_PCT", "DREB_PCT", "REB_PCT", "TS_PCT", "EFG_PCT", "PACE", "PIE",
]
advanced_small = advanced_raw[[c for c in ADV_KEEP if c in advanced_raw.columns]].copy() if not advanced_raw.empty else pd.DataFrame()

# Merging Base + Advanced + Position into one master player-season table
master = base_raw.merge(advanced_small, on=["PLAYER_ID", "SEASON"], how="left") if not base_raw.empty else pd.DataFrame()
if not position_raw.empty and not master.empty:
    master = master.merge(position_raw, on=["PLAYER_ID", "SEASON"], how="left")

# Saving the master dataframe to disk
if not master.empty:
    master.to_csv(PROCESSED_DATA_DIR / "master_player_season_raw.csv", index=False)
    print(master.shape)
    master.head()

(5387, 80)
